In [ ]:
import numpy as np
import pandas as pd
class GrowthFirmValuationModel:
  def __init__(self, current_backlog, baseline_capex, capex_efficiency):
    self.current_backlog = current_backlog
    self.baseline_capex = baseline_capex
    self.capex_efficiency = capex_efficiency
  def simulate_cash_flows(self, years, cancellation_prob, volatility, simulations=1000):
    np.random.seed(42)
    all_sim_results = []
    for _ in range(simulations):
      backlog = self.current_backlog
      cash_flows = []
      for t in range(1, years + 1):
        if np.random.rand() < cancellation_prob:
          backlog *= np.random.uniform(0.7, 0.9)
        conversion_rate = 0.20 * (1 +(self.capex_efficiency *0.05))
        revenue_from_backlog = backlog * conversion_rate
        adoption_shock = np.random.normal(0, volatility)
        final_revenue = revenue_from_backlog * (1 + adoption_shock)
        net_cash_flow = final_revenue - self.baseline_capex
        cash_flows.append(net_cash_flow)
        backlog = (backlog - revenue_from_backlog) * 1.10
      all_sim_results.append(cash_flows)
    return pd.DataFrame(all_sim_results)
model = GrowthFirmValuationModel(current_backlog=500000000, baseline_capex=50000000, capex_efficiency=1.5)
simulated_data = model.simulate_cash_flows(years=5, cancellation_prob=0.15, volatility=0.10)
print(f"mean expected year 5 cash flow: ${simulated_data[4].mean():,.2f}")

mean expected year 5 cash flow: $908,057.51


In [ ]:
cfar_5pct = simulated_data[4].quantile(0.5)
print(f"5% cash flow at risk (worst case): ${cfar_5pct:,.2f}")

5% cash flow at risk (worst case): $1,627,492.21


In [ ]:
insolvency_risk = (simulated_data[4] < 0).mean() * 100
print(f"probability of negative cash flow in year 5: {insolvency_risk: .2f}%")

probability of negative cash flow in year 5:  45.40%


In [ ]:
worst_5pct_outcomes = simulated_data[4][simulated_data[4] <= cfar_5pct]
expected_shortfall = worst_5pct_outcomes.mean()
print(f"expected shortfall (average of worst 5% outcomes): ${expected_shortfall:,.2f}")

expected shortfall (average of worst 5% outcomes): $-7,619,619.97


In [ ]:
def calculate_npv_distribution(sim_df, discount_rate=0.10):
  npv_results = []
  for index, row in sim_df.iterrows():
    npv = 0
    for t, cash_flow in enumerate(row):
      npv += cash_flow / (1 + discount_rate) ** (t + 1)
    npv_results.append(npv)
  return pd.Series(npv_results)
npv_distribution = calculate_npv_distribution(simulated_data)
print(f"expected intrinsic value (mean npv): ${npv_distribution.mean():,.2f}")
print(f"standard deviation of npv: ${npv_distribution.std():,.2f}")
#

expected intrinsic value (mean npv): $105,151,536.88
standard deviation of npv: $33,714,145.02


In [ ]:
def generate_risk_report(self, sim_df, discount_rate=0.10):
  year_5_data = sim_df[sim_df.columns[-1]]
  cfar_5 = year_5_data.quantile(0.05)
  burn_prob = (year_5_data < 0).mean() * 100
  expected_shortfall = year_5_data[year_5_data <= cfar_5].mean()
  print(" Quantatative Risk Report")
  print("===========================")
  print(f"mean expected final year cash flow: ${year_5_data.mean():,.2f}")
  print(f"5% cash flow at risk (worst case): ${cfar_5:,.2f}")
  print(f"expected shortfall (tail risk avg) : ${expected_shortfall:,.2f}")
  print(f"probability of negative cash flow in year 5: {burn_prob: .2f}%")
  print("===========================")

